# Clase 3 · 06 · Columnar: por qué Parquet y Delta son rápidos para analítica

Desde la clase 1 usamos Parquet y Delta sin mirar adentro. Una base **columnar** guarda juntos los valores de cada columna en lugar de cada fila. Para preguntas analíticas (*"monto promedio de las transacciones grandes"*) permite:

1. **Leer sólo las columnas necesarias** (*column pruning*).
2. **Comprimir mejor**: valores del mismo tipo, uno al lado del otro.
3. **Saltear bloques enteros** usando estadísticas min/max (*data skipping*).

Vamos a escribir las mismas transacciones como CSV, JSON y Parquet y comparar.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

## 1. Fila vs columna, dibujado

Las mismas cuatro transacciones en disco. El color indica la columna. Con el borde negro se marcan los valores que necesita `SELECT avg(amount)`.

In [ ]:
sample = (spark.table(transactions).orderBy("transaction_id")
          .select("transaction_id", "customer_id", "amount", "payment_channel").limit(4).toPandas())
columns = list(sample.columns)
palette = dict(zip(columns, ["#bfdbfe", "#bbf7d0", "#fde68a", "#fbcfe8"]))
row_layout = [(c, r) for r in range(len(sample)) for c in columns]
col_layout = [(c, r) for c in columns for r in range(len(sample))]

fig, axes = plt.subplots(2, 1, figsize=(14, 3.6))
for ax, layout, title in [(axes[0], row_layout, "Orientado a filas (CSV, JSON, bases OLTP)"),
                          (axes[1], col_layout, "Orientado a columnas (Parquet, Delta)")]:
    for i, (c, r) in enumerate(layout):
        needed = c == "amount"
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=palette[c], edgecolor="black" if needed else "white", lw=2.5 if needed else 1))
        ax.text(i + 0.5, 0.5, str(sample.loc[r, c])[:8], ha="center", va="center", fontsize=7)
    ax.set_xlim(0, len(layout))
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=10)
fig.tight_layout()
plt.show()

## 2. Mismos datos, tres formatos

Para que las diferencias se vean, replicamos `silver_transactions` `COPIES` veces (cambiando el `transaction_id`).

In [ ]:
COPIES = 10
columnar_path = f"{lab_path}/columnar"
data = (spark.table(transactions).drop("processed_at")
        .crossJoin(spark.range(COPIES).withColumnRenamed("id", "copy"))
        .withColumn("transaction_id", F.col("transaction_id") + F.col("copy") * 100_000_000)
        .drop("copy"))
paths = {"CSV": f"{columnar_path}/csv", "JSON": f"{columnar_path}/json", "Parquet": f"{columnar_path}/parquet"}
data.coalesce(1).write.mode("overwrite").option("header", True).csv(paths["CSV"])
data.coalesce(1).write.mode("overwrite").json(paths["JSON"])
data.coalesce(1).write.mode("overwrite").parquet(paths["Parquet"])

def data_files(path):
    files = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            files += data_files(item.path)
        elif not item.name.startswith(("_", ".")):
            files.append(item)
    return files

sizes_mb = {name: sum(f.size for f in data_files(p)) / 1024 ** 2 for name, p in paths.items()}
rows = spark.read.parquet(paths["Parquet"]).count()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(list(sizes_mb), list(sizes_mb.values()), color=["#f59e0b", "#ef4444", "#16a34a"])
for i, v in enumerate(sizes_mb.values()):
    ax.text(v, i, f" {v:.1f} MB", va="center")
ax.set_xlabel("Tamaño en disco (MB)")
ax.set_title(f"{rows:,} transacciones en tres formatos")
plt.show()

## 3. Leer sólo lo necesario

Ejecutamos la misma consulta sobre cada formato y miramos el plan físico. En Parquet, `ReadSchema` muestra que sólo se lee la columna `amount`; en CSV y JSON hay que leer y separar cada línea completa aunque después se descarten las otras columnas.

In [ ]:
import contextlib
import io
import time

schema = spark.read.parquet(paths["Parquet"]).schema
readers = {
    "CSV": lambda: spark.read.schema(schema).option("header", True).csv(paths["CSV"]),
    "JSON": lambda: spark.read.schema(schema).json(paths["JSON"]),
    "Parquet": lambda: spark.read.parquet(paths["Parquet"]),
}

def query(df):
    return df.where("amount > 1900").agg(F.avg("amount").alias("avg_amount"), F.count("*").alias("n"))

plan_buffer = io.StringIO()
with contextlib.redirect_stdout(plan_buffer):
    query(readers["Parquet"]()).explain(mode="formatted")
scan_lines = [l.strip() for l in plan_buffer.getvalue().splitlines()
              if any(word in l for word in ("Scan", "ReadSchema", "PushedFilters", "DataFilters"))]
print("\n".join(scan_lines) or plan_buffer.getvalue())

timings = {}
for name, reader in readers.items():
    runs = []
    for _ in range(3):
        start = time.perf_counter()
        result = query(reader()).first()
        runs.append(time.perf_counter() - start)
    timings[name] = min(runs)
    print(f"{name:8s} avg_amount={result.avg_amount:.2f}  n={result.n:,}  mejor tiempo={timings[name]:.2f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(list(timings), list(timings.values()), color=["#f59e0b", "#ef4444", "#16a34a"])
ax.set_xlabel("Segundos (mejor de 3)")
ax.set_title("avg(amount) WHERE amount > 1900")
plt.show()

> Los tiempos en serverless incluyen overhead fijo y varían entre ejecuciones; con datos chicos la diferencia puede ser modesta. El tamaño y el plan son evidencia más estable que el cronómetro.

## 4. Adentro de un archivo Parquet

Parquet guarda al final del archivo un **footer** con el esquema y, por cada *row group* y columna, la compresión y las estadísticas min/max. Lo leemos con `pyarrow`.

In [ ]:
import pyarrow.parquet as pq

def local_path(file_info):
    return file_info.path.replace("dbfs:", "")

def stat(column, attr):
    """min/max del footer; algunos tipos (p. ej. timestamps) no tienen estadísticas legibles con pyarrow."""
    try:
        stats = column.statistics
        return getattr(stats, attr) if stats is not None and stats.has_min_max else None
    except Exception:
        return "(no disponible)"

parquet_file = data_files(paths["Parquet"])[0]
metadata = pq.ParquetFile(local_path(parquet_file)).metadata
print(f"Filas: {metadata.num_rows:,} · row groups: {metadata.num_row_groups} · columnas: {metadata.num_columns}")
rg = metadata.row_group(0)
display(pd.DataFrame([{
    "columna": rg.column(i).path_in_schema,
    "compresión": rg.column(i).compression,
    "MB comprimido": round(rg.column(i).total_compressed_size / 1024 ** 2, 2),
    "MB sin comprimir": round(rg.column(i).total_uncompressed_size / 1024 ** 2, 2),
    "min": stat(rg.column(i), "min"),
    "max": stat(rg.column(i), "max"),
} for i in range(rg.num_columns)]))

## 5. Data skipping: ordenar importa

Escribimos los datos en 8 archivos de dos formas: **al azar** y **ordenados por `amount`**. Dibujamos el rango min–max de `amount` de cada archivo. Un filtro `amount > 1900` puede saltear todo archivo cuyo máximo sea menor que 1900. Delta hace esto automáticamente con las estadísticas de su `_delta_log` (y con `OPTIMIZE ... ZORDER BY`).

In [ ]:
skip_data = data.select("transaction_id", F.col("amount").cast("double").alias("amount"))   # double: estadísticas legibles en cualquier versión de pyarrow
layouts = {
    "al azar": (skip_data.repartition(8), f"{columnar_path}/parquet_random"),
    "ordenado por amount": (skip_data.repartitionByRange(8, "amount"), f"{columnar_path}/parquet_sorted"),
}
THRESHOLD = 1900
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharex=True)
for ax, (name, (frame, path)) in zip(axes, layouts.items()):
    frame.write.mode("overwrite").parquet(path)
    ranges = []
    for f in data_files(path):
        md_ = pq.ParquetFile(local_path(f)).metadata
        col = [i for i in range(md_.num_columns) if md_.row_group(0).column(i).path_in_schema == "amount"][0]
        stats = [md_.row_group(g).column(col).statistics for g in range(md_.num_row_groups)]
        ranges.append((float(min(s.min for s in stats)), float(max(s.max for s in stats))))
    ranges.sort()
    skipped = sum(hi < THRESHOLD for _, hi in ranges)
    for i, (lo, hi) in enumerate(ranges):
        ax.plot([lo, hi], [i, i], lw=6, color="#cbd5e1" if hi < THRESHOLD else "#2563eb")
    ax.axvline(THRESHOLD, color="#dc2626", ls="--", label=f"amount > {THRESHOLD}")
    ax.set_title(f"{name}: se pueden saltear {skipped} de {len(ranges)} archivos")
    ax.set_xlabel("Rango de amount en el archivo")
    ax.set_yticks(range(len(ranges)), [f"archivo {i + 1}" for i in range(len(ranges))])
    ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 6. Delta = Parquet + log de transacciones

Las tablas de las clases 1 y 2 son Delta: archivos Parquet más un `_delta_log` con las versiones, el esquema y las estadísticas por archivo.

In [ ]:
display(spark.sql(f"DESCRIBE DETAIL {transactions}").select("format", "numFiles", "sizeInBytes", "partitionColumns"))

## 7. Resumen: compresión y columnas leídas

Comparamos el plan de la consulta anterior (sólo `amount`) con una que necesita **todas** las columnas.

In [ ]:
print(f"Parquet ocupa {sizes_mb['CSV'] / sizes_mb['Parquet']:.1f} veces menos que CSV y {sizes_mb['JSON'] / sizes_mb['Parquet']:.1f} veces menos que JSON")

def read_schema(df):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode="formatted")
    return [l.strip() for l in buffer.getvalue().splitlines() if "ReadSchema" in l]

parquet_df = readers["Parquet"]()
one_column = query(parquet_df)
all_columns = parquet_df.where("amount > 1900").agg(*[F.max(c).alias(c) for c in parquet_df.columns])
print("\nConsulta sobre amount:      ", read_schema(one_column))
print("Consulta sobre todas:        ", read_schema(all_columns))